# Notebook 5: Validation and Sample Queries
Runs pass/fail checks with failed-row samples and executes demo join queries.

In [ ]:
from datetime import datetime, timedelta
from pyspark.sql import functions as F

window_start = datetime.now() - timedelta(days=90)

provider = spark.table('provider')
patient = spark.table('patient')
appointment = spark.table('appointment')
diagnosis = spark.table('diagnosis')
prescription = spark.table('prescription')

In [ ]:
results = []

def add_result(name, passed, detail):
    results.append({
        'check_name': name,
        'status': 'PASS' if passed else 'FAIL',
        'detail': detail
    })

add_result('provider_count', provider.count() == 20, 'Expected 20 providers')
add_result('patient_count', patient.count() == 50, 'Expected 50 patients')
add_result('appointment_count', appointment.count() == 100, 'Expected 100 appointments')
add_result('diagnosis_count', diagnosis.count() == 20, 'Expected 20 diagnoses')
add_result('prescription_count', prescription.count() == 30, 'Expected 30 prescriptions')

add_result('provider_id_unique', provider.select('providerId').distinct().count() == 20, 'providerId must be unique')
add_result('patient_id_unique', patient.select('patientId').distinct().count() == 50, 'patientId must be unique')
add_result('appointment_id_unique', appointment.select('appointmentId').distinct().count() == 100, 'appointmentId must be unique')
add_result('diagnosis_id_unique', diagnosis.select('diagnosisId').distinct().count() == 20, 'diagnosisId must be unique')
add_result('rx_number_unique', prescription.select('rxNumber').distinct().count() == 30, 'rxNumber must be unique')

add_result('all_appointments_completed', appointment.filter(F.col('status') != 'completed').count() == 0, 'All status values should be completed')
add_result('appointment_window_3_months', appointment.filter((F.col('scheduledTime') < F.lit(window_start)) | (F.col('scheduledTime') > F.current_timestamp())).count() == 0, 'Appointments must be within past 3 months')

patient_no_appt = patient.alias('p').join(appointment.alias('a'), on='patientId', how='left_anti')
provider_no_dx = provider.alias('p').join(diagnosis.alias('d'), on='providerId', how='left_anti')
provider_no_rx = provider.alias('p').join(prescription.alias('r'), on='providerId', how='left_anti')
dx_no_rx = diagnosis.alias('d').join(prescription.alias('r'), on='diagnosisId', how='left_anti')

add_result('every_patient_has_appointment', patient_no_appt.count() == 0, 'No patient should be missing appointments')
add_result('every_provider_has_diagnosis', provider_no_dx.count() == 0, 'No provider should be missing diagnoses')
add_result('every_provider_has_prescription', provider_no_rx.count() == 0, 'No provider should be missing prescriptions')
add_result('every_diagnosis_has_prescription', dx_no_rx.count() == 0, 'No diagnosis should be untreated')

summary_df = spark.createDataFrame(results)
display(summary_df.orderBy('status', 'check_name'))

In [ ]:
# Failed-row samples (shown only if the corresponding check fails).
if patient_no_appt.count() > 0:
    print('Failed sample: patients without appointments')
    display(patient_no_appt.limit(20))

if provider_no_dx.count() > 0:
    print('Failed sample: providers without diagnoses')
    display(provider_no_dx.limit(20))

if provider_no_rx.count() > 0:
    print('Failed sample: providers without prescriptions')
    display(provider_no_rx.limit(20))

if dx_no_rx.count() > 0:
    print('Failed sample: diagnoses without prescriptions')
    display(dx_no_rx.limit(20))

In [ ]:
# Sample query 1: Patient activity summary
q1 = spark.sql("""
SELECT p.patientId, p.name, COUNT(a.appointmentId) AS appointment_count
FROM patient p
LEFT JOIN appointment a ON p.patientId = a.patientId
GROUP BY p.patientId, p.name
ORDER BY appointment_count DESC, p.patientId
""")
display(q1.limit(20))

In [ ]:
# Sample query 2: Provider workload and outcomes
q2 = spark.sql("""
SELECT pr.providerId, pr.name,
       COUNT(DISTINCT a.appointmentId) AS appt_count,
       COUNT(DISTINCT d.diagnosisId) AS dx_count,
       COUNT(DISTINCT rx.rxNumber) AS rx_count
FROM provider pr
LEFT JOIN appointment a ON pr.providerId = a.providerId
LEFT JOIN diagnosis d ON pr.providerId = d.providerId
LEFT JOIN prescription rx ON pr.providerId = rx.providerId
GROUP BY pr.providerId, pr.name
ORDER BY rx_count DESC, dx_count DESC, appt_count DESC
""")
display(q2)

In [ ]:
# Sample query 3: Diagnosis to treatment chain
q3 = spark.sql("""
SELECT d.diagnosisId, d.icdCode, d.description, d.severity,
       COUNT(rx.rxNumber) AS treatment_count
FROM diagnosis d
LEFT JOIN prescription rx ON d.diagnosisId = rx.diagnosisId
GROUP BY d.diagnosisId, d.icdCode, d.description, d.severity
ORDER BY treatment_count DESC, d.diagnosisId
""")
display(q3)

In [ ]:
# Sample query 4: Monthly completed appointments by specialty
q4 = spark.sql("""
SELECT date_format(a.scheduledTime, 'yyyy-MM') AS year_month,
       pr.specialty,
       COUNT(*) AS completed_appointments
FROM appointment a
JOIN provider pr ON a.providerId = pr.providerId
WHERE a.status = 'completed'
GROUP BY date_format(a.scheduledTime, 'yyyy-MM'), pr.specialty
ORDER BY year_month, completed_appointments DESC
""")
display(q4)

In [ ]:
# Sample query 5: Full patient-provider-diagnosis-prescription chain
q5 = spark.sql("""
SELECT p.patientId, p.name AS patient_name,
       a.appointmentId, a.scheduledTime,
       pr.name AS provider_name,
       d.diagnosisId, d.icdCode,
       rx.rxNumber, rx.medication
FROM patient p
JOIN appointment a ON p.patientId = a.patientId
JOIN provider pr ON a.providerId = pr.providerId
LEFT JOIN diagnosis d ON d.patientId = p.patientId AND d.providerId = pr.providerId
LEFT JOIN prescription rx ON rx.diagnosisId = d.diagnosisId
ORDER BY a.scheduledTime DESC
""")
display(q5.limit(50))

In [ ]:
# Persist validation summary for repeatable demo evidence.
summary_with_ts = summary_df.withColumn('validated_at', F.current_timestamp())
summary_with_ts.write.mode('overwrite').format('delta').saveAsTable('validation_summary')
display(spark.table('validation_summary').orderBy('status', 'check_name'))